# Data Analysis with fg-data-profiling

> Tanvir Anjom Siddique <br>
> Visit my portfolio: [tanvirsweb.github.io](https://tanvirsweb.github.io)


In [6]:
# If you want the exact environment packages installed by the current Python interpreter, I recommend:
# import sys
# !{sys.executable} -m pip freeze > requirements.txt
import sys
!{sys.executable} -m pip install ydata-profiling

  Using cached filetype-1.2.0-py2.py3-none-any.whl.metadata (6.5 kB)
  Using cached multimethod-1.12-py3-none-any.whl.metadata (9.6 kB)
  Using cached ImageHash-4.3.2-py2.py3-none-any.whl.metadata (8.4 kB)
  Using cached dacite-1.9.2-py3-none-any.whl.metadata (17 kB)
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.0 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.0 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.0 MB 572.3 kB/s eta 0:00:14
   -- ------------------------------------- 0.5/8.0 MB 572.3 kB/s eta 0:00:14
   --- ------------------------------------ 0.8/8.0 MB 701.5 kB/s eta 0:00:11
   ----- ---------------------------------- 1.0/8.0 MB 748.7 kB/s eta 0:00:10
   ----- ---------------------------------- 1.0/8.0 MB 748.7 kB/s eta 0:00:10
   ------ --------------------------------- 1.3/8.0 MB 692.8 kB/s 

In [7]:
# uncomment and run below commands
!python --version
# !python -m ensurepip --upgrade
# !python.exe -m pip install --upgrade pip
# !python -m pip install -q fg-data-profiling pandas openpyxl

Python 3.13.9


In [8]:
!pip freeze > requirements.txt

In [9]:
def data_profiling_app():
    import tkinter as tk
    from tkinter import filedialog, messagebox
    import pandas as pd
    from ydata_profiling import ProfileReport
    import os, webbrowser, threading

    last_df = None
    last_report_file = None

    BG, CARD, BLUE, HOVER = "#F5F7FA", "#FFFFFF", "#2563EB", "#1D4ED8"
    TEXT, SECONDARY, BORDER = "#1F2937", "#6B7280", "#E5E7EB"
    SUCCESS, WARNING = "#16A34A", "#D97706"

    def status(text, color=SECONDARY):
        status_label.config(text=text, fg=color)

    def load_file():
        nonlocal last_df, last_report_file
        path = filedialog.askopenfilename(title="Select Dataset", filetypes=[("CSV files", "*.csv"), ("Excel files", "*.xlsx *.xls"), ("All supported files", "*.csv *.xlsx *.xls")])
        if not path: return
        try:
            status("Loading dataset...", WARNING)
            if path.lower().endswith(".csv"): df = pd.read_csv(path)
            elif path.lower().endswith((".xlsx", ".xls")): df = pd.read_excel(path)
            else: raise ValueError("Unsupported file type.")
            last_df, last_report_file = df, None
            file_label.config(text=os.path.basename(path))
            rows_label.config(text=f"{len(df):,}")
            cols_label.config(text=f"{len(df.columns):,}")
            missing_label.config(text=f"{df.isna().sum().sum():,}")
            dataset_card.pack(fill="x", padx=30, pady=(10, 15))
            generate_btn.config(state="normal")
            view_btn.config(state="disabled")
            export_btn.config(state="disabled")
            status("Dataset loaded successfully", SUCCESS)
        except Exception as e:
            status("Failed to load dataset", WARNING)
            messagebox.showerror("Loading Error", f"Could not load the dataset:\n\n{e}")

    def generate_report():
        nonlocal last_report_file
        if last_df is None:
            messagebox.showwarning("No Dataset", "Please load a dataset first.")
            return
        generate_btn.config(state="disabled")
        view_btn.config(state="disabled")
        export_btn.config(state="disabled")
        status("Generating profiling report...", WARNING)
        def task():
            nonlocal last_report_file
            try:
                last_report_file = os.path.join(os.getcwd(), "temp_profile.html")
                ProfileReport(last_df, title="Data Profiling Report").to_file(last_report_file)
                root.after(0, report_ready)
            except Exception as e:
                root.after(0, lambda: report_error(e))
        threading.Thread(target=task, daemon=True).start()

    def report_ready():
        status("Report generated successfully", SUCCESS)
        generate_btn.config(state="normal")
        view_btn.config(state="normal")
        export_btn.config(state="normal")
        open_report()

    def report_error(error):
        status("Failed to generate report", WARNING)
        generate_btn.config(state="normal")
        messagebox.showerror("Report Error", f"Could not generate profiling report:\n\n{error}")

    def open_report():
        if not last_report_file or not os.path.exists(last_report_file):
            messagebox.showwarning("No Report", "Please generate a report first.")
            return
        webbrowser.open(os.path.abspath(last_report_file))

    def export_report():
        if last_df is None:
            messagebox.showwarning("No Dataset", "Please load a dataset first.")
            return
        path = filedialog.asksaveasfilename(title="Export Profiling Report", defaultextension=".html", filetypes=[("HTML files", "*.html")])
        if not path: return
        export_btn.config(state="disabled")
        status("Exporting report...", WARNING)
        def task():
            try:
                ProfileReport(last_df, title="Data Profiling Report").to_file(path)
                root.after(0, lambda: export_success(path))
            except Exception as e:
                root.after(0, lambda: export_error(e))
        threading.Thread(target=task, daemon=True).start()

    def export_success(path):
        export_btn.config(state="normal")
        status("Report exported successfully", SUCCESS)
        messagebox.showinfo("Export Successful", f"Report saved successfully:\n\n{path}")

    def export_error(error):
        export_btn.config(state="normal")
        status("Export failed", WARNING)
        messagebox.showerror("Export Error", f"Could not export report:\n\n{error}")

    def create_stat(parent, title, value, column):
        frame = tk.Frame(parent, bg="#F8FAFC")
        frame.grid(row=0, column=column, sticky="ew", padx=5)
        parent.columnconfigure(column, weight=1)
        value_label = tk.Label(frame, text=value, font=("Segoe UI", 20, "bold"), bg="#F8FAFC", fg=BLUE)
        value_label.pack(pady=(12, 0))
        tk.Label(frame, text=title, font=("Segoe UI", 9), bg="#F8FAFC", fg=SECONDARY).pack(pady=(0, 12))
        return value_label

    root = tk.Tk()
    root.title("Data Profiling Studio")
    root.configure(bg=BG)
    sw, sh = root.winfo_screenwidth(), root.winfo_screenheight()
    w, h = int(sw * 0.75), int(sh * 0.75)
    root.geometry(f"{w}x{h}+{(sw-w)//2}+{(sh-h)//2}")
    root.minsize(800, 600)

    header = tk.Frame(root, bg=BLUE, height=100)
    header.pack(fill="x")
    header.pack_propagate(False)
    tk.Label(header, text="Data Profiling Studio", font=("Segoe UI", 24, "bold"), bg=BLUE, fg="white").pack(anchor="w", padx=30, pady=(18, 0))
    tk.Label(header, text="Explore, analyze and understand your CSV or Excel datasets", font=("Segoe UI", 11), bg=BLUE, fg="#DBEAFE").pack(anchor="w", padx=32)

    content = tk.Frame(root, bg=BG)
    content.pack(fill="both", expand=True)

    upload_card = tk.Frame(content, bg=CARD, highlightbackground=BORDER, highlightthickness=1)
    upload_card.pack(fill="x", padx=30, pady=(25, 10))
    tk.Label(upload_card, text="Start Data Analysis", font=("Segoe UI", 16, "bold"), bg=CARD, fg=TEXT).pack(anchor="w", padx=25, pady=(20, 2))
    tk.Label(upload_card, text="Select a CSV or Excel file to generate a detailed profiling report.", font=("Segoe UI", 10), bg=CARD, fg=SECONDARY).pack(anchor="w", padx=25, pady=(0, 15))
    tk.Button(upload_card, text="＋  Load Dataset", command=load_file, font=("Segoe UI", 11, "bold"), bg=BLUE, fg="white", activebackground=HOVER, activeforeground="white", relief="flat", cursor="hand2", padx=20, pady=10).pack(anchor="w", padx=25, pady=(0, 20))

    dataset_card = tk.Frame(content, bg=CARD, highlightbackground=BORDER, highlightthickness=1)
    tk.Label(dataset_card, text="Dataset Information", font=("Segoe UI", 15, "bold"), bg=CARD, fg=TEXT).pack(anchor="w", padx=25, pady=(18, 8))
    file_label = tk.Label(dataset_card, text="No file selected", font=("Segoe UI", 10), bg=CARD, fg=SECONDARY)
    file_label.pack(anchor="w", padx=25)

    stats = tk.Frame(dataset_card, bg=CARD)
    stats.pack(fill="x", padx=25, pady=15)
    rows_label = create_stat(stats, "Rows", "0", 0)
    cols_label = create_stat(stats, "Columns", "0", 1)
    missing_label = create_stat(stats, "Missing Values", "0", 2)

    actions = tk.Frame(content, bg=BG)
    actions.pack(fill="x", padx=30, pady=10)

    generate_btn = tk.Button(actions, text="Generate Report", command=generate_report, font=("Segoe UI", 10, "bold"), bg=BLUE, fg="white", activebackground=HOVER, relief="flat", cursor="hand2", padx=20, pady=10, state="disabled")
    generate_btn.pack(side="left", padx=(0, 10))

    view_btn = tk.Button(actions, text="View Report", command=open_report, font=("Segoe UI", 10), bg=CARD, fg=TEXT, activebackground=BORDER, relief="flat", cursor="hand2", padx=20, pady=10, state="disabled")
    view_btn.pack(side="left", padx=(0, 10))

    export_btn = tk.Button(actions, text="Export HTML", command=export_report, font=("Segoe UI", 10), bg=CARD, fg=TEXT, activebackground=BORDER, relief="flat", cursor="hand2", padx=20, pady=10, state="disabled")
    export_btn.pack(side="left")

    status_frame = tk.Frame(root, bg=CARD, height=40, highlightbackground=BORDER, highlightthickness=1)
    status_frame.pack(fill="x", side="bottom")
    status_frame.pack_propagate(False)
    status_label = tk.Label(status_frame, text="Ready — Load a dataset to begin", font=("Segoe UI", 9), bg=CARD, fg=SECONDARY)
    status_label.pack(anchor="w", padx=20, pady=10)

    root.mainloop()
    


In [10]:
data_profiling_app()

C:\Users\mdfah\AppData\Local\Temp\ipykernel_2412\2704649762.py:5: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 5/5 [00:00<00:00, 7266.64it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

## Data Analysis with ydata-profiling


In [11]:
# !python -m pip install ydata-profiling
# !python -m conda update -n base -c defaults conda

!echo "open anaconda promt: run as administrator > run below command"
# !conda install -c conda-forge ydata-profiling


"open anaconda promt: run as administrator > run below command"


In [12]:
# !conda activate base
# !pip install pywebview

In [13]:
def app():
    import pandas as pd
    import seaborn as sns
    from ydata_profiling import ProfileReport

    df = sns.load_dataset("titanic")
    # Create profiling report
    profile = ProfileReport(df, title="Profiling Report", explorative=True)
    profile.to_file(output_file='MyReport.html')

    # Display the report directly in notebook
    # profile.to_widgets()
    profile.to_notebook_iframe()
    
# app()

## GUI with ydata_profiling

In [14]:
def dataAnalyzer():    
    import tkinter as tk
    from tkinter import filedialog, messagebox
    import pandas as pd
    from ydata_profiling import ProfileReport
    import os
    import webview  # pip install pywebview

    last_df = None
    last_report_file = None

    def load_file():
        global last_df, last_report_file
        file_path = filedialog.askopenfilename(
            title="Select CSV or Excel File",
            filetypes=[("CSV files", "*.csv"), ("Excel files", "*.xlsx *.xls")]
        )
        if not file_path:
            return

        try:
            if file_path.endswith(".csv"):
                df = pd.read_csv(file_path)
            elif file_path.endswith((".xlsx", ".xls")):
                df = pd.read_excel(file_path)
            else:
                messagebox.showerror("Error", "Unsupported file type.")
                return

            last_df = df
            last_report_file = None

            # Generate temporary profiling report for viewing
            temp_html = os.path.join(os.getcwd(), "temp_profile.html")
            profile = ProfileReport(df, title="Profiling Report", explorative=True)
            profile.to_file(temp_html)
            last_report_file = temp_html

            # Open HTML inside the app
            webview.create_window("Profiling Report", temp_html)
            webview.start()

        except Exception as e:
            messagebox.showerror("Error", f"An error occurred:\n{e}")

    def export_report():
        global last_df
        if last_df is None:
            messagebox.showwarning("Warning", "No data to export.")
            return

        save_path = filedialog.asksaveasfilename(
            defaultextension=".html",
            filetypes=[("HTML files", "*.html")],
            title="Export Profiling Report As"
        )
        if save_path:
            try:
                profile = ProfileReport(last_df, title="Profiling Report", explorative=True)
                profile.to_file(save_path)
                messagebox.showinfo("Saved", f"Report exported to:\n{save_path}")
            except Exception as e:
                messagebox.showerror("Error", f"Could not export report:\n{e}")

    # Tkinter GUI
    root = tk.Tk()
    root.title("CSV/Excel Profiling Tool")

    # Get screen width and height
    screen_width = root.winfo_screenwidth()
    screen_height = root.winfo_screenheight()

    # Set window to 90% of screen size
    width = int(screen_width * 0.9)
    height = int(screen_height * 0.9)

    # Center the window
    x = (screen_width - width) // 2
    y = (screen_height - height) // 2
    root.geometry(f"{width}x{height}+{x}+{y}")

    # Optional: make window resizable
    root.resizable(True, True)

    # Widgets
    btn_load = tk.Button(root, text="Load File", command=load_file, width=20)
    btn_load.pack(pady=10)

    btn_export = tk.Button(root, text="Export as HTML", command=export_report, width=20, state="normal")
    btn_export.pack(pady=10)

    root.mainloop()

# dataAnalyzer()